In [ ]:
import pandas as pd
import commons as c
import plotly.express as px
import plotly.graph_objects as go

# Get datasets

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [ ]:
df_equiv['label'] = "Equivalent mutant"
df_normal['label'] = "Non-Equivalent mutant"
df = pd.concat([df_equiv, df_normal], ignore_index=True)

# Get Box Plots

In [ ]:
category_names = {
        'Qubits_number': 'Number of qubits', 
        'gates': 'Number of gates', 
        'depth': 'Circuit depth',
        'Algorithm': 'Algorithm name', 
        'Input_type': 'Type of input', 
        'Output_type': 'Type of output',
        'Gate_type': 'Mutated gate', 
        'Operator': 'Mutation operator', 
        'Relative_position': 'Relative position of the mutation'
    }

In [ ]:
def print_box_plot(df, cat, distance, output_folder, file_name, median_line=False, width=2000):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column   
    
    # Create the box plot
    fig = px.box(
        df, 
        y=distance, 
        x=cat, 
        color="label", 
        category_orders={
            cat: cat_range,
            "label": ["Equivalent mutant", "Non-Equivalent mutant"]
        },
        # title="Boxplot of Distance by noise model and program type",
        labels={cat: category_names[cat], distance: "Distance", 'label': "Legend"},
        points=False,
        boxmode="group"
    ) 
    
    # Compute medians for each category and true_label
    median_values = df.groupby([cat, "label"])[distance].median().reset_index()
    
    if median_line:
        # Define colors matching the boxplot
        colors = {"Equivalent mutant": "blue", "Non-Equivalent mutant": "red"}
        
        for label in median_values["label"].unique():
            subset = median_values[median_values["label"] == label]
            fig.add_trace(go.Scatter(
                x=subset[cat], 
                y=subset[distance], 
                mode='lines',
                name=f"Median - {label}",
                line=dict(color=colors[label], dash='dot') 
            ))
    
    # Adjust layout for better visualization
    fig.update_layout(
        xaxis=dict(tickmode="array", tickvals=cat_range)
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], width=width)

In [ ]:
def category_plot(df, m):
    category_groups = [
        ('Qubits_number', "RQ2_1", True, 2000), # 7
        ('gates', "RQ2_1", True, 3000), # 65
        ('depth', "RQ2_1", True, 3000), # 42
        ('Algorithm', "RQ2_2", False, 2000), # 5
        ('Input_type', "RQ2_2", False, 1000), # 2
        ('Output_type', "RQ2_2", False, 1000), # 2
        ('Gate_type', "RQ2_3", False, 1000), # 2
        ('Operator', "RQ2_3", False, 1000), # 3
        ('Relative_position', "RQ2_3", True, 2000) # 5
    ]
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]

        for cat, subfolder, showmedian, width in category_groups:
            selected_columns = df_metric[[cat, 'label', 'ideal_distance', 'noisy_distance']]
            file_name = f'{hw}_{cat}'
                
            for distance_type in ['noisy_distance', 'ideal_distance']:
                variant = 'noisy' if distance_type == 'noisy_distance' else 'ideal'
                output_folder = f'results/RQ2/{subfolder}/{variant}/{m}'
                print_box_plot(selected_columns, cat, distance_type, output_folder, file_name, showmedian, width)


In [ ]:
m = "T"
category_plot(df, m)

m = "H"
category_plot(df, m)